In [1]:
from deeppavlov import configs
from deeppavlov import build_model
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

# Загрузка toxic_comments.csv (EN)


In [2]:
import csv

with open('toxic_comments.csv', 'r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    df = pd.DataFrame(reader)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113082 entries, 0 to 113081
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0           113082 non-null  object
 1   text    113082 non-null  object
 2   toxic   113081 non-null  object
dtypes: object(3)
memory usage: 2.6+ MB


## Меняю тип данных toxic на int


In [4]:
df['toxic'] = df['toxic'].fillna('0')
df['toxic'] = df['toxic'].astype('int64')
#df = df.drop(df.columns[0], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113082 entries, 0 to 113081
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0           113082 non-null  object
 1   text    113082 non-null  object
 2   toxic   113082 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.6+ MB


## Лемматизация EN


In [29]:
#nltk.download('wordnet')
#nltk.download('omw-1.4') 
#nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_english(text):
    if pd.isna(text):
        return ""

    
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
    words = text.split()
    
    return ' '.join([lemmatizer.lemmatize(word) for word in words])


df['lemmatized'] = df['text'].apply(preprocess_english)


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113082 entries, 0 to 113081
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0               113082 non-null  object
 1   text        113082 non-null  object
 2   toxic       113082 non-null  int64 
 3   lemmatized  113082 non-null  object
dtypes: int64(1), object(3)
memory usage: 3.5+ MB


In [8]:
#df = df.drop(df.columns[3], axis=1)
#df = df.drop(df.columns[0], axis=1)

## Векторизация EN

In [30]:
count_tf_idf = TfidfVectorizer(stop_words=list(stop_words))
tf_idf = count_tf_idf.fit_transform(df['lemmatized'])

print(f"Размер матрицы: {tf_idf.shape}")
print(f"Количество признаков: {len(count_tf_idf.get_feature_names_out())}")

Размер матрицы: (113082, 129279)
Количество признаков: 129279


# DeepPavlov EN

In [41]:
model_en = build_model('insults_kaggle_bert', download=True)


sample_size = 20
test_df_en = df.head(sample_size).copy()


test_df_en['pred_original'] = model_en(test_df_en['text'].tolist())


test_df_en['pred_lemm'] = model_en(test_df_en['lemmatized'].tolist())


result_df_en = test_df_en[['text', 'pred_original', 'lemmatized', 'pred_lemm']].copy()
result_df_en.columns = ['Оригинал', 'Тон (ориг)', 'Леммы', 'Тон (леммы)']


display(result_df_en)


2026-03-12 17:40:12.641 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/datasets/insults_data.tar.gz download because of matching hashes
2026-03-12 17:40:15.88 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/insults_kaggle_torch_bert_v5.tar.gz download because of matching hashes
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.

,Оригинал,Тон (ориг),Леммы,Тон (леммы)
0,Explanation\nWhy the edits made under my usern...,Not Insult,explanation why the edits made under my userna...,Not Insult
1,D'aww! He matches this background colour I'm s...,Not Insult,d aww he match this background colour i m seem...,Not Insult
2,"Hey man, I'm really not trying to edit war. It...",Not Insult,hey man i m really not trying to edit war it s...,Not Insult
3,"""\nMore\nI can't make any real suggestions on ...",Not Insult,more i can t make any real suggestion on impro...,Not Insult
4,"You, sir, are my hero. Any chance you remember...",Not Insult,you sir are my hero any chance you remember wh...,Not Insult
5,"""\n\nCongratulations from me as well, use the ...",Not Insult,congratulation from me a well use the tool wel...,Not Insult
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,Insult,cocksucker before you piss around on my work,Insult
7,Your vandalism to the Matt Shirvington article...,Not Insult,your vandalism to the matt shirvington article...,Insult
8,Sorry if the word 'nonsense' was offensive to ...,Not Insult,sorry if the word nonsense wa offensive to you...,Not Insult
9,alignment on this subject and which are contra...,Not Insult,alignment on this subject and which are contra...,Not Insult


# Загрузка rusentitweet_full (RU)


In [14]:
import csv

with open('rusentitweet_full.csv', 'r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    df_ru = pd.DataFrame(reader)

In [15]:
df_ru.head()

,,text,label,id
0,0,@varlamov @McFaul На,skip,1327934765807308801
1,1,велл они всё равно что мусор так что ничего с...,negative,1252943181387350017
2,2,"""трезвая жизнь какая-то такая стрёмная""\n(с) а...",negative,1323610669061677056
3,3,Ой какие неожиданные результаты 🤭 https://t.co...,neutral,1336231661160247297
4,4,@Shvonder_chief @dimsmirnov175 На заборе тоже ...,neutral,1292421736454127617


In [20]:
df_ru.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13392 entries, 0 to 13391
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    13392 non-null  object
 1   label   13392 non-null  object
dtypes: object(2)
memory usage: 209.4+ KB


In [19]:
#df_ru = df_ru.drop('id', axis=1) 
#df_ru = df_ru.drop(df_ru.columns[0], axis=1)

## Лематизация (RU)

In [33]:
import pymorphy2
morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text):
    if pd.isna(text):
        return ""

    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', str(text).lower())

    text = re.sub(r'(.)\1{2,}', r'\1', text)

    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    return ' '.join([morph.parse(word)[0].normal_form for word in words])

df_ru['lemmatized'] = df_ru['text'].apply(lemmatize_ru)

In [38]:
print("Оригинал:", df_ru['text'].iloc[:10])
print("Леммы:", df_ru['lemmatized'].iloc[:10])

Оригинал: 0                                 @varlamov @McFaul На
1    велл они  всё равно что мусор так что ничего с...
2    "трезвая жизнь какая-то такая стрёмная"\n(с) а...
3    Ой какие неожиданные результаты 🤭 https://t.co...
4    @Shvonder_chief @dimsmirnov175 На заборе тоже ...
5        @idkwhht мы тоже мебельная компания уджина😳😳😳
6    Счастья здоровья 10 классникам https://t.co/M9...
7                                   @dntbliev НЕ ПАЛИ.
8                     @BTS_twt ты такой красивый 😭😭😭🥺💓
9                        @Ladyzchensk Цыган , хуле ...
Name: text, dtype: object
Леммы: 0                                                   на
1    велла они всё равно что мусор так что ничего с...
2        трезвый жизнь какой то такой стрёмный с артём
3                       ой какой неожиданный результат
4    на забор тоже написать а там другой о сбор ден...
5                    мы тоже мебельный компания уджина
6                            счастие здоровье классник
7                     

# Векторизация (RU)

In [ ]:
nltk.download('stopwords')
stop_words_ru = set(stopwords.words('russian'))


count_tf_idf_ru = TfidfVectorizer(stop_words=list(stop_words_ru))
tf_idf_ru = count_tf_idf_ru.fit_transform(df_ru['lemmatized'])

print(f"Размер матрицы: {tf_idf_ru.shape}")
print(f"Количество признаков: {len(count_tf_idf_ru.get_feature_names_out())}")




In [39]:
print("\nПримеры признаков:")
print(count_tf_idf_ru.get_feature_names_out()[20:])


Примеры признаков:
['абрикос' 'абсент' 'абсолютно' ... 'ёпишг' 'ёпттвоюмать' 'ёстоить']


# DeepPavlov (Ru)

In [40]:
model_ru = build_model('rusentiment_bert', download=True)

sample_size = 20
test_df = df_ru.head(sample_size).copy()


test_df['pred_original'] = model_ru(test_df['text'].tolist())
test_df['pred_lemm'] = model_ru(test_df['lemmatized'].tolist())

result_df = test_df[['text', 'pred_original', 'lemmatized', 'pred_lemm']].copy()
result_df.columns = ['Оригинал', 'Тон (ориг)', 'Леммы', 'Тон (леммы)']

display(result_df)



2026-03-12 17:34:05.409 INFO in 'deeppavlov.core.data.utils'['utils'] at line 97: Downloading from http://files.deeppavlov.ai/v1/classifiers/rusentiment_bert/rusentiment_bert_torch.tar.gz to C:\Users\danii\.deeppavlov\models\classifiers\rusentiment_bert_torch.tar.gz
100%|█████████████████████████████████████████████████████████████████████████████| 1.34G/1.34G [02:02<00:00, 11.0MB/s]
2026-03-12 17:36:08.872 INFO in 'deeppavlov.core.data.utils'['utils'] at line 284: Extracting C:\Users\danii\.deeppavlov\models\classifiers\rusentiment_bert_torch.tar.gz archive into C:\Users\danii\.deeppavlov\models\classifiers\rusentiment_bert_torch
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\

,Оригинал,Тон (ориг),Леммы,Тон (леммы)
0,@varlamov @McFaul На,neutral,на,neutral
1,велл они всё равно что мусор так что ничего с...,negative,велла они всё равно что мусор так что ничего с...,negative
2,"""трезвая жизнь какая-то такая стрёмная""\n(с) а...",neutral,трезвый жизнь какой то такой стрёмный с артём,negative
3,Ой какие неожиданные результаты 🤭 https://t.co...,neutral,ой какой неожиданный результат,neutral
4,@Shvonder_chief @dimsmirnov175 На заборе тоже ...,neutral,на забор тоже написать а там другой о сбор ден...,neutral
5,@idkwhht мы тоже мебельная компания уджина😳😳😳,positive,мы тоже мебельный компания уджина,neutral
6,Счастья здоровья 10 классникам https://t.co/M9...,speech,счастие здоровье классник,positive
7,@dntbliev НЕ ПАЛИ.,neutral,не пасть,neutral
8,@BTS_twt ты такой красивый 😭😭😭🥺💓,positive,ты такой красивый,positive
9,"@Ladyzchensk Цыган , хуле ...",negative,цыган хула,skip
